<a href="https://colab.research.google.com/github/AKRaj2023/AKRaj2023/blob/main/Copy_of_Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# homoplot.py
# Usage: C:\SiH4.mat
import sys
import numpy as np
import matplotlib.pyplot as plt

def try_scipy_load(path):
    try:
        from scipy import io
        m = io.loadmat(path, squeeze_me=True, struct_as_record=False)
        # remove MATLAB meta keys
        return {k:v for k,v in m.items() if not k.startswith('__')}
    except Exception as e:
        return None

def try_h5py_load(path):
    try:
        import h5py
        d = {}
        f = h5py.File(path, 'r')
        def read_item(k, obj):
            if isinstance(obj, h5py.Dataset):
                try:
                    d[k] = obj[()]
                except Exception:
                    d[k] = None
            else:
                for sub in obj:
                    read_item(k + '/' + sub, obj[sub])
        for key in f:
            read_item(key, f[key])
        f.close()
        return d
    except Exception:
        return None

def inspect(mat):
    print("Variables found (name : type, shape):")
    for k,v in mat.items():
        try:
            print(f" - {k} : {type(v).__name__}, shape={getattr(v,'shape',None)}")
        except Exception:
            print(f" - {k} : (unprintable)")
    print()

def find_candidates(mat):
    names = {k.lower():k for k in mat.keys()}
    # guesses
    grid_vars = []
    orb_vars = []
    energy_var = None
    occ_var = None
    for lname, orig in names.items():
        if any(x in lname for x in ('grid','xyz','xgrid','rgrid','coords')) and hasattr(mat[orig], 'shape') and mat[orig].ndim==2:
            grid_vars.append(orig)
        if any(x in lname for x in ('orb','psi','mo','orbital','molecular_orbital')) and hasattr(mat[orig], 'shape'):
            # 3D grid or (npoints, norb) etc
            orb_vars.append(orig)
        if 'energy' in lname or 'energies' in lname or 'eigs' in lname:
            energy_var = orig
        if 'occup' in lname or 'occ' in lname:
            occ_var = orig
        if any(x in lname for x in ('density','rho')) and hasattr(mat[orig],'shape') and mat[orig].ndim==3:
            # density grid exists
            orb_vars.append(orig)
    return grid_vars, orb_vars, energy_var, occ_var

def choose_homo_index(mat, energy_var, occ_var, orb_var):
    # If occupations exist, use them to find last occupied orbital
    if occ_var is not None:
        occ = np.asarray(mat[occ_var]).ravel()
        idxs = np.where(occ>0.5)[0]
        if idxs.size>0:
            homo = idxs.max()
            print(f"Found occupations; choosing HOMO index = {homo} (occupation>0.5).")
            return homo
    if energy_var is not None:
        energies = np.asarray(mat[energy_var]).ravel()
        # assume negative energies occupied or count by below Fermi if present
        # fallback: choose highest energy with index less than count of electrons/2 if available
        # pick max index with energy <= median of energies if uncertain
        try:
            # guess occupancy: if energies sorted ascending
            # find highest index where energy <= median_of_low_half
            sorted_idx = np.argsort(energies)
            energies_sorted = energies[sorted_idx]
            # heuristics: find gap between occupied and virtual (largest gap)
            gaps = np.diff(energies_sorted)
            if gaps.size>0:
                cut = np.argmax(gaps)
                homo_sorted_idx = sorted_idx[cut]
                print(f"Found energies; heuristic HOMO index = {homo_sorted_idx}.")
                return int(homo_sorted_idx)
        except Exception:
            pass
    # If orbitals exist and are 3D grid with shape (nx,ny,nz,norb) or (norb,nx,ny,nz)
    a = mat[orb_var]
    a = np.asarray(a)
    if a.ndim==4:
        # guess last occupied is half or user-provided; pick middle orbital
        norb = a.shape[-1]
        homo = norb//2 - 1
        print(f"Found 4D orbital grid; guessing HOMO index = {homo}.")
        return homo
    # fallback choose last index
    try:
        norb = a.shape[0] if a.ndim>0 else 0
        if norb>1:
            homo = norb-1
            print(f"Fallback: choosing HOMO index = {homo}.")
            return homo
    except Exception:
        pass
    print("Unable to determine HOMO index automatically.")
    return None

def plot_2d_slice(orb_grid, title='HOMO (central slice)', cmap='seismic'):
    # orb_grid assumed 3D array (nx,ny,nz)
    nx,ny,nz = orb_grid.shape
    zc = nz//2
    sl = orb_grid[:,:,zc]
    plt.figure(figsize=(6,5))
    plt.imshow(sl.T, origin='lower', cmap=cmap, interpolation='nearest')
    plt.colorbar(label='orbital amplitude')
    plt.title(title + f' (z slice {zc})')
    plt.xlabel('x index'); plt.ylabel('y index')
    plt.tight_layout()

def plot_isosurface(grid3d, level=None, title='HOMO isosurface'):
    # requires scikit-image
    try:
        from skimage import measure
    except Exception:
        print("skimage not installed: skip isosurface. Install scikit-image to enable.")
        return
    if level is None:
        # choose level at 50% of max abs
        level = 0.5 * np.max(np.abs(grid3d))
    verts, faces, normals, values = measure.marching_cubes(grid3d, level=level)
    # plot via mpl 3d
    from mpl_toolkits.mplot3d import Axes3D  # noqa
    fig = plt.figure(figsize=(8,6))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot_trisurf(verts[:, 0], verts[:,1], faces, verts[:, 2],
                    cmap='Spectral', linewidth=0.1, alpha=0.7)
    ax.set_title(f"{title} (isosurface level={level:.3g})")
    plt.tight_layout()

def main(path):
    mat = try_scipy_load(path)
    if mat is None:
        mat = try_h5py_load(path)
    if mat is None:
        print("Failed to read the .mat file with scipy.io or h5py.")
        return
    inspect(mat)
    grid_vars, orb_vars, energy_var, occ_var = find_candidates(mat)
    print("Guessed grid-like variables:", grid_vars)
    print("Guessed orbital-like variables:", orb_vars)
    print("Energy var:", energy_var, "Occ var:", occ_var)
    if len(orb_vars)==0:
        print("No orbital-like 3D arrays found. If your file stores MO coefficients (not grid), we need basis/grid info to build real-space orbitals.")
        print("If you see variable names, paste them here and I will adapt.")
        return
    # pick first orbital variable
    orb_name = orb_vars[0]
    print("Using orbital variable:", orb_name)
    orb = np.asarray(mat[orb_name])
    # possible shapes:
    # (nx,ny,nz,norb)  OR (norb, nx, ny, nz) OR (nx,ny,nz) (single orbital) OR (npoints, norb)
    if orb.ndim==3:
        # single orbital grid
        homo_grid = orb
        print("Single 3D array found; assuming it's the HOMO.")
    elif orb.ndim==4:
        # determine axis with norb
        # heuristic: last axis is norb if small compared to others
        axes = orb.shape
        # try last axis as orbital
        if axes[-1] <= 200:  # arbitrary
            homo_idx = choose_homo_index(mat, energy_var, occ_var, orb_name)
            if homo_idx is None:
                print("Please specify HOMO index. Using index 0 as fallback.")
                homo_idx = 0
            homo_grid = orb[..., int(homo_idx)]
            print(f"Extracted orbital index {homo_idx} from last axis.")
        else:
            # maybe first axis is norb
            homo_idx = choose_homo_index(mat, energy_var, occ_var, orb_name)
            if homo_idx is None:
                homo_idx = 0
            homo_grid = orb[int(homo_idx), ...]
            print(f"Extracted orbital index {homo_idx} from first axis.")
    else:
        print("Orbital variable has unsupported number of dimensions:", orb.ndim)
        return

    # show 2D central slice
    plot_2d_slice(homo_grid, title=f'HOMO from {orb_name}')
    # attempt isosurface
    try:
        plot_isosurface(homo_grid, level=0.2*np.max(np.abs(homo_grid)))
    except Exception as e:
        print("Isosurface plotting failed:", e)
    plt.show()

if __name__ == '__main__':
    if len(sys.argv)<2:
        print("Usage: python homoplot.py path/to/file.mat")
    else:
        main(sys.argv[1])

Failed to read the .mat file with scipy.io or h5py.
